# CS-681 Project: Text Result Figures
Runs on CPU. Attach the STCSv1 notebook output (reports.csv) as input.

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score

OUT = "/kaggle/working"
plt.rcParams.update({"font.size": 11})
df = pd.read_csv(glob.glob("/kaggle/input/**/reports.csv", recursive=True)[0])

GRADE_WORDS = [(r"\bproliferative\b", 4), (r"\bsevere\b", 3), (r"\bmoderate\b", 2), (r"\bmild\b", 1),
    (r"\bno (?:obvious |apparent |clear |definite |significant |visible )*(?:signs? of |evidence of )?(?:diabetic retinopathy|dr)\b", 0),
    (r"\bnormal (?:retina|fundus)\b", 0)]
CONTEXT = r"(retinopathy|npdr|\bdr\b|severity|stage|grade)"

def grade_structured(text):
    m = re.search(r"SEVERITY GRADE:\s*([0-4])", str(text))
    return int(m.group(1)) if m else np.nan

def grade_zero_shot(text):
    """Same rule-based extractor as the evaluation notebook."""
    t = re.sub(r"non[- ]?proliferative", "npdr", str(text).lower().replace("*", ""))
    heads = [m.end() for m in re.finditer(r"\n[^\n]*(assessment|severity)[^\n]*:\s*\n", t)]
    section = t[heads[-1]:] if heads else t
    section = re.split(r"\n[^\n]*(important considerations|disclaimer|recommendation|next steps)[^\n]*:", section)[0]
    for sent in re.split(r"(?<=[.!?\n])\s+", section):
        if re.search(CONTEXT, sent):
            found = sorted({g for p, g in GRADE_WORDS if re.search(p, sent)})
            if found:
                return found[0] if len(found) == 1 else np.nan
    return np.nan

df["pred_grade"] = np.where(df["prompt_type"] == "structured",
                            df["report"].apply(grade_structured), df["report"].apply(grade_zero_shot))
df["words"] = df["report"].str.split().str.len()
print(df.groupby("prompt_type")["pred_grade"].apply(lambda s: s.notna().mean()))

## Figure 1: main comparison of the two prompting strategies

In [ ]:
def metrics(g):
    ok = g.dropna(subset=["pred_grade"])
    y, p = ok["true_grade"].astype(int), ok["pred_grade"].astype(int)
    return {"Clear grade\n(extraction rate)": len(ok) / len(g),
            "Exact accuracy\n(strict)": (p.values == y.values).sum() / len(g),
            "QWK\n(clear reports)": cohen_kappa_score(y, p, weights="quadratic")}

m = pd.DataFrame({k: metrics(g) for k, g in df.groupby("prompt_type")})
ax = m[["structured", "zero_shot"]].plot(kind="bar", figsize=(7, 4.2), color=["#4C72B0", "#DD8452"], rot=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=9)
ax.set_ylim(0, 1.12); ax.set_ylabel("Score")
ax.set_title("Structured template vs zero-shot prompting (n = 250 each)")
ax.legend(["Structured template", "Zero-shot"])
plt.tight_layout(); plt.savefig(f"{OUT}/fig_text_comparison.png", dpi=200); plt.show()
print(m.round(3))

## Figure 2: confusion matrices

In [ ]:
from sklearn.metrics import confusion_matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (name, title) in zip(axes, [("structured", "Structured template"), ("zero_shot", "Zero-shot (clear reports only)")]):
    ok = df[(df.prompt_type == name) & df.pred_grade.notna()]
    cm = confusion_matrix(ok.true_grade, ok.pred_grade.astype(int), labels=range(5))
    ax.imshow(cm, cmap="Blues")
    for i in range(5):
        for j in range(5):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xlabel("Predicted grade"); ax.set_ylabel("True grade"); ax.set_title(f"{title} (n={len(ok)})")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_confusion_matrices.png", dpi=200); plt.show()

## Figure 3: report length and generation time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
order = ["structured", "zero_shot"]; labels = ["Structured", "Zero-shot"]; colors = ["#4C72B0", "#DD8452"]
for ax, col, ylab in zip(axes, ["words", "seconds"], ["Words per report", "Seconds per report"]):
    vals = [df[df.prompt_type == k][col] for k in order]
    bp = ax.boxplot(vals, patch_artist=True)
    ax.set_xticks([1, 2], labels)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
    ax.set_ylabel(ylab)
axes[0].set_title("Report length"); axes[1].set_title("Generation time (Kaggle T4 x2)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_length_time.png", dpi=200); plt.show()

## Figure 4: internal consistency of structured reports

In [ ]:
FIELDS = ["Microaneurysms", "Hemorrhages", "Hard exudates", "Cotton wool spots",
          "Venous beading or IRMA", "Neovascularization", "Vitreous or preretinal hemorrhage"]
KEYWORDS = {"Neovascularization": "neovascular", "Hemorrhages": "hemorrhage", "Hard exudates": "exudate",
            "Cotton wool spots": "cotton wool", "Venous beading or IRMA": "venous beading", "Microaneurysms": "microaneurysm"}

def findings(text):
    out = {}
    for f in FIELDS:
        m = re.search(rf"-\s*{re.escape(f)}:\s*(Present|Absent)", str(text), re.I)
        out[f] = None if not m else m.group(1).lower() == "present"
    return out

def issues(row):
    f, g = findings(row["report"]), row["pred_grade"]
    pdr = bool(f["Neovascularization"] or f["Vitreous or preretinal hemorrhage"])
    m = re.search(r"SUMMARY:\s*(.+)", str(row["report"]), re.S)
    summ = m.group(1).lower() if m else ""
    return pd.Series({
        "Grade 4 without\nPDR signs": g == 4 and not pdr,
        "Summary contradicts\nFINDINGS": any(f[k] is False and kw in summ for k, kw in KEYWORDS.items()),
        "Any inconsistency": (g == 4 and not pdr) or any(f[k] is False and kw in summ for k, kw in KEYWORDS.items()),
    })

st = df[df.prompt_type == "structured"].copy()
iss = st.apply(issues, axis=1).mean() * 100
ax = iss.plot(kind="bar", color=["#C44E52", "#8172B2", "#937860"], rot=0, figsize=(6.5, 3.8))
ax.bar_label(ax.containers[0], fmt="%.1f%%")
ax.set_ylabel("% of structured reports"); ax.set_ylim(0, max(iss) * 1.3)
ax.set_title("Internal consistency checks (structured, n = 250)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_consistency.png", dpi=200); plt.show()

## Analysis: accuracy by model-reported image quality (structured)

In [ ]:
st["quality"] = st["report"].str.extract(r"IMAGE QUALITY:\s*(\w+)", expand=False)
rows = []
for qual, g in st.groupby("quality"):
    rows.append({"quality": qual, "n": len(g),
                 "exact_accuracy": (g.pred_grade == g.true_grade).mean(),
                 "within_1": ((g.pred_grade - g.true_grade).abs() <= 1).mean(),
                 "true_grade_distribution": g.true_grade.value_counts().sort_index().to_dict()})
q = pd.DataFrame(rows).set_index("quality").round(3)
print(q.to_string())
q.to_csv(f"{OUT}/accuracy_by_image_quality.csv")

## Per-grade accuracy (structured)

In [ ]:
rows = []
for grade, g in st.groupby("true_grade"):
    rows.append({"true_grade": grade,
                 "exact_accuracy": (g.pred_grade == g.true_grade).mean(),
                 "mean_predicted_grade": g.pred_grade.mean(),
                 "over_graded": (g.pred_grade > g.true_grade).mean(),
                 "under_graded": (g.pred_grade < g.true_grade).mean()})
pg = pd.DataFrame(rows).set_index("true_grade").round(3)
print(pg)
pg.to_csv(f"{OUT}/structured_per_grade.csv")